In [ ]:
!pip install langchain chromadb faiss-cpu openai tiktoken langchain_openai langchain-community wikipedia

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 144.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.6 MB/s eta 0:00:00
   ━━━━━

Wikipedia Retriever

In [ ]:
from langchain_community.retrievers import WikipediaRetriever


In [ ]:
# Initialize the retriever (optional: set language and top_k)
retriever = WikipediaRetriever(top_k_results=2, lang="en")

In [ ]:
# Define your query
query= "the geopolitical history of india and pakistan from the perspective of a chinese"

# Get relevant Wikipedia documents
docs = retriever.invoke(query)

In [ ]:
docs

In [ ]:
# Print retrieved content
for i, doc in enumerate(docs):
  print(f"\n ---Result {i+1}---")
  print(f"Content: {doc.page_content}...") # truncate for display


Vector Store Retriever

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document


In [ ]:
# Step 1 : Your source documents
documents = [
    Document(page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose"),
    Document(page_content="Leo DiCaprio gets lost in a dream within a dream within a dream within a ..."),
    Document(page_content="A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea"),
    Document(page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them"),
]

In [ ]:
# Step 2 : Initialize embedding model
embedding_model = OpenAIEmbeddings()

# Step 3 : Create  Chroma vector store in memory
vectorstore = Chroma.from_documents(
    documents = documents,
    embedding = embedding_model,
    collection_name = 'my_collection'
    )

In [ ]:
# Step 4: Convert vectorstore into a retriever
retriever = vectorstore.as_retriever(
    search_kwargs = {'k': 2}
    )

In [ ]:
query = " what is Chroma used for ?"
results = retriever.invoke(query)

In [ ]:
for i, doc in enumerate(results):
  print(f"\n ---Result {i+1}---")
  print(doc.page_content)

MMR

Stratergy-Maximal Marginal Relevance (not similar)

In [ ]:
# Sample documents
docs = [
    Document(page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose"),
    Document(page_content="Leo DiCaprio gets lost in a dream within a dream within a dream within a ..."),
    Document(page_content="A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea"),
    Document(page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them"),
    Document(page_content="Three men walk into the Zone, three men walk out of the Zone, three men walk into the hourglass ..."),
    Document(page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them"),
]

In [ ]:
from langchain_community.vectorstores import FAISS

# Initialize OpenAI embeddings
embedding_medel = OpenAIEmbeddings()

# Step 2: Create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents = docs,
    embedding = embedding_model
)

In [ ]:
# Enable MMR in the retriever
retriever = vectorstore.as_retriever(
    search_type = 'mmr',                    # <---- This enables MMR
    search_kwargs = {'k': 3, "lambda_mult": 0.5}    # k = top results , lambda_mult = relevance-diversity balance (0-1)
)

In [ ]:
query = "what is langchain?"
results = retriever.invoke(query)

In [ ]:
for i, doc in enumerate (results):
  print (f"\n --- Result {i+1}---")
  print(doc.page_content)

Multi-Query Retriever (ambiguous)

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import OpenAIEmbeddings
from lasngchain_core.documents import Document
from langchain_openai import OpenAI

In [ ]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose", metadata = {"source": "H1"}),
    Document(page_content="Leo DiCaprio gets lost in a dream within a dream within a dream within a ...", metadata = {"source": "H2"}),
    Document(page_content="A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea", metadata = {"source": "H3"}),
    Document(page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them", metadata = {"source": "H4"}),
    Document(page_content="Three men walk into the Zone, three men walk out of the Zone, three men walk into the hourglass ...", metadata = {"source": "H5"}),
    Document(page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them", metadata = {"source": "I1"}),
    Document(page_content="Three men walk into the Zone, three men walk out of the Zone, three men walk into the hourglass ...", metadata = {"source": "I2"}),
    Document(page_content="The men walk into the Zone three men walk out of the Zone, three men walk into the hourglass ...", metadata = {"source": "I3"}),
    Document(page_content="Three men walk into the Zone, three men walk out of the Zone, three men walk into the hourglass ...", metadata = {"source": "I4"}),
    Document(page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them", metadata = {"source": "I5"}),

]

In [ ]:
# Initialize OpenAI embeddings
embedding_model = OpenAIEmbeddings()

# Create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents = all_docs,
    embedding = embedding_model
)


In [ ]:
# Create retrievers
similarity_retriever = vectorstore.as_retriever(
    search_type = 'similarity',
    search_kwargs = {'k': 5}
)

In [ ]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever = vectorstore.as_retriever(search_kwargs = {'k': 5}),
    llm = ChatOpenAI(model = "gpt-3.5-turbo")
)

In [ ]:
# Query
query = " How to improve energy levels and maintain balance?"

In [ ]:
# Retrieve results
similarity_results = similarity_retriever.invoke(query)
multiquery_results = multiquery_retriever.invoke(query)

In [ ]:
for i, result in enumerate(similarity_results):
  print(f"\n --- Result {i+1} ---")
  print(result.page_content)
  print(result.metadata)

print("*"*150)

for i, result in enumerate(multiquery_results):
  print(f"\n --- Result {i+1} ---")
  print(result.page_content)
  print(result.metadata)


Contextual Compression Retriever (Important line)

In [ ]:
from langchain_community.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.retrievers.document_compressors import LLMChainExtractor
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings , ChatOpenAI
from langchain_core.documents import Document

In [ ]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose", metadata = {"source": "Doc1"}),
    Document(page_content="Leo DiCaprio gets lost in a dream within a dream within a dream within a ...", metadata = {"source": "Doc2"}),
    Document(page_content="A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea", metadata = {"source": "Doc3"}),
    Document(page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them", metadata = {"source": "Doc4"}),
]

In [ ]:
# Create a FAISS vector store from the documents
embedding_model = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embedding_model)

In [ ]:
base_retriever = vectorstore.as_retriever(
    search_kwargs = {'k': 5}
)

In [ ]:
# Set up the compressor using an LLM
llm = ChatOpenAI(model = "gpt-3.5-turbo")
compressor = LLMChainExtractor.from_llm(llm)

In [ ]:
# Create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor = compressor,
    base_retriever = base_retriever
)

In [ ]:
# Query the retriever
query = " what is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [ ]:
for i, doc in enumerate (compressed_results):
  print(f"\n --- Result {i+1} ---")
  print(doc.page_content)
  print(doc.metadata)